We want to know if a cell has consistent tuning across conditions or between homing/escpae and explore

1. compute tuning
2. compute linear sift tuning 
3. if curve is significant in both conditions
4. bootstrap for similarity metric (subsample many times from both the homing and exploration period and compute the firing by distance. Then I take pairs of subsampled tuning curves and compute some metric of similarity (e.g. difference in the preferred distance or cosine similarity) and that gives me a whole distribution of the metrics. If 0 falls outside the 95% of that distribution then the two curves are different.) 

In [1]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept, JAL3_22aug

from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

#JAL3_7sept, JAL3_4sept, JAL3_1sept, JAL3_25aug, JAL3_22aug,
# 
experiments_objects = [JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, # JAL6_flip7_1apr, # this session is sus
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_flip4_10may, JAL8_14may]

In [ ]:
%load_ext autoreload
from JR_test_scripts.escape.functions.escape_utils import load, load_homing
from behave_analysis.utils.creating_directories import make_directory
from JR_test_scripts.escape.functions.escape_data_loading_funcs import extract_explore_periods
from JR_test_scripts.escape.functions.escape_tuning_funcs import tuning_method_no_trials_with_pool
from behave_analysis.utils.creating_directories import make_directory
from JR_test_scripts.escape.functions.escape_plotting_funcs import new_plot_linear_shift
from scipy.ndimage import gaussian_filter1d

import numpy as np
import matplotlib.pyplot as plt
import gc
%matplotlib inline

In [3]:
"""Linear shift significance for tuning curves
THIS IS THE GOOD ONE!!"""
%autoreload 2
"""Compute real statistics (on the full session)"""

# compression = ['bird_dist_shelter','bird_dist_first_goal', 'full_distance_shelter']
compression = ['bird_dist_shelter']
Nbins = 25

for exp in experiments_objects[16:]:

    print(exp.nick_name + '_' + exp.experiment_date)
    
    session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, escape, outofshelter = load(exp)
    ons, offs, homie = load_homing(session, len(behave))
    fcm = gaussian_filter1d(frame_by_cluster_matrix, 2, axis = 0)

    del frame_by_cluster_matrix
    gc.collect()

    for comp in compression:
        
        nickname = exp.nick_name + '_' + exp.experiment_date
        exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp

        if comp == 'bird_dist_shelter':
            # var = compute_dist_shelt(x_pos, y_pos, cond=np.zeros_like(x_pos), session=session)
            # bins = np.arange(0,np.amax(var),np.amax(var)/Nbins)
            bins = np.append(np.arange(0,925,925/Nbins), 925)

        # the real stat
        var, escape_matrix, cond = extract_explore_periods(session,
                                                                fcm,
                                                                behave,
                                                                y_pos, x_pos,
                                                                bar,
                                                                barflip,
                                                                comp,
                                                                homie,
                                                                escape,
                                                                outofshelter,
                                                                bins = bins)

        # initialize vars
        n_neur = fcm.shape[1]
        n_cond = len(np.unique(cond))

        y_fitted_full, R_full, fr_full, params_full = tuning_method_no_trials_with_pool(var, 
                                                                                        escape_matrix, 
                                                                                        cond, 
                                                                                        Nbins, n_cond, n_neur, fitting = False)

        del var, escape_matrix, cond
        gc.collect()

        """Compute real statistics (on the central third of each condition)"""

        # setting up the shifts
        T = np.shape(fcm)[0]
        central_chunk = T/3
        N = int((T - central_chunk)/2)
        min_step = 120
        step = 400
        step_n = 100
        shifts_one_sided = np.arange(min_step,min_step+((step_n/2)*step), step)

        shelter = np.where(bar == True)[0][0]
        bar_in = np.where(barflip == True)[0][0]
        mid_shelter = [int(shelter/3), int((shelter/3)*2)]
        mid_bar = [int(shelter+((bar_in - shelter)/3)), int(shelter+(((bar_in - shelter)/3)*2))]
        mid_flip = [int(bar_in+((len(bar) - bar_in)/3)), int(bar_in+(((len(bar) - bar_in)/3)*2))]
        shift_vector = np.zeros(len(bar))
        shift_vector[mid_shelter[0]:mid_shelter[1]] = 1
        shift_vector[mid_bar[0]:mid_bar[1]] = 1
        shift_vector[mid_flip[0]:mid_flip[1]] = 1
        shift_vector = shift_vector.astype(bool)

        # make sure we're not shifting our of range
        shifts_left = shifts_one_sided[shifts_one_sided < mid_shelter[0]]
        shifts_right = shifts_one_sided[(shifts_one_sided + mid_flip[1]) < len(bar)]
        # now double them so we go in both directions
        shifts = np.sort(np.hstack((shifts_right,-shifts_left)))

        # the real stat
        var, escape_matrix, cond = extract_explore_periods(session,
                                                            fcm[shift_vector,:],
                                                            behave[shift_vector],
                                                            y_pos[shift_vector], x_pos[shift_vector],
                                                            bar[shift_vector],
                                                            barflip[shift_vector],
                                                            comp,
                                                            homie[shift_vector],
                                                            escape[shift_vector],
                                                            outofshelter[shift_vector],
                                                            bins = bins)

        y_fitted_real, R_real, fr_real, params_real = tuning_method_no_trials_with_pool(var, 
                                                                                        escape_matrix, 
                                                                                        cond, 
                                                                                        Nbins, n_cond, n_neur, fitting = False)

        del var, escape_matrix, cond
        gc.collect()

        """Compute shifted statistics"""

        # initialize variables for output
        y_fitted_shift = np.full((step_n, n_cond, n_neur, Nbins), np.nan) # conditions x neurons x n_bins
        R_shift = np.zeros((step_n,n_neur, n_cond)) # neurons x conditions
        params_shifts = np.zeros((step_n,n_neur, n_cond, 6)) # neurons x conditions
        fr_shift = np.full((step_n, n_cond, n_neur, Nbins), np.nan)

        for s_idx, s in enumerate(shifts):
            shifted_vec = np.roll(shift_vector,int(s))
            var, escape_matrix, cond = extract_explore_periods(session,
                                                                fcm[shifted_vec,:],
                                                                behave[shift_vector],
                                                                y_pos[shift_vector], x_pos[shift_vector],
                                                                bar[shift_vector],
                                                                barflip[shift_vector],
                                                                comp,
                                                                homie[shift_vector],
                                                                escape[shift_vector],
                                                                outofshelter[shift_vector],
                                                                bins = bins)
            
            y_fitted_shift[s_idx,:,:,:], R_shift[s_idx,:,:], fr_shift[s_idx,:,:,:], params_shifts[s_idx,:,:,:] = tuning_method_no_trials_with_pool(var, 
                                                                                                                                        escape_matrix, 
                                                                                                                                        cond, 
                                                                                                                                        Nbins, n_cond, n_neur, fitting = False)

        dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves_explore/")
        saving_file = dump_path + exp_nickname + '_Tuning_' + str(Nbins) + 'bins'
        np.savez(saving_file, 
                R_shift=R_shift, params_shifts=params_shifts, y_fitted_shift=y_fitted_shift, 
                fr_shift=fr_shift, 
                R_full=R_full, params_full=params_full, y_fitted_full=y_fitted_full, 
                fr_full=fr_full, full_reliability=[],
                R_real=R_real, params_real=params_real, y_fitted_real=y_fitted_real, 
                fr_real=fr_real)

        """Plot linear shift and real stats"""
        colors = ['#228B22','#FF8C00','#008B8B']
        c_names = ['shelter_only', 'barrier', 'barrier_flipped']
        nickname = exp.nick_name + '_' + exp.experiment_date
        exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp + '_' + str(Nbins) + 'bins'
        dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves_explore/" + nickname + "/" + exp_nickname)

        new_plot_linear_shift(y_fitted_shift, y_fitted_real, y_fitted_full, params_shifts, params_real, fr_full, fr_shift, params_full, comp, n_neur, n_cond, colors, c_names, dump_path, name = '_new')

        del (var, escape_matrix, cond, 
        y_fitted_shift, R_shift, fr_shift, params_shifts,
        fr_full, y_fitted_full, R_full, params_full,
        y_fitted_real, R_real, fr_real, params_real)
        gc.collect()

JAL008_2024_04_29


2025-03-14 08:14:41.243 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


JAL008_2024_05_7


2025-03-14 08:59:26.301 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


JAL008_2024_05_10


2025-03-14 09:42:18.610 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


JAL008_2024_05_14


2025-03-14 10:11:38.164 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...
